In [10]:
#turn this into .py files later.
import os
os.environ["TESSDATA_PREFIX"] = r"C:\Program Files\Tesseract-OCR\tessdata"

import pymupdf.layout
import pymupdf4llm
import ollama
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re


In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=300,
    add_start_index=True
)

client = chromadb.Client()
collection = client.get_or_create_collection(name="local_pdf_rag")

In [14]:
pdf_folder = "data/"

def ingest_pdf(file_path):
    md_text = pymupdf4llm.to_markdown(file_path, use_ocr=True, ocr_language="eng")
    
    clean_text = re.sub(r"\*\*==> picture.*?<==\*\*", "", md_text, flags=re.DOTALL)
    
    chunks = text_splitter.create_documents([clean_text])
    
    file_name = os.path.basename(file_path)
    
    for i, chunk in enumerate(chunks):
        emb = ollama.embeddings(
            model="qwen3-embedding:0.6b",
            prompt=chunk.page_content
        )["embedding"]

        collection.add(
            ids=[f"{file_name}_chunk_{i}"],
            embeddings=[emb],
            documents=[chunk.page_content],
            metadatas=[{
                "source": file_name,
                "chunk": i
            }]
        )

for file in os.listdir(pdf_folder):
    if file.endswith(".pdf"):
        ingest_pdf(os.path.join(pdf_folder, file))

In [16]:
#RETRIEVE & GENERATE: The RAG Function
def local_rag(query):
    query_emb = ollama.embeddings(model="qwen3-embedding:0.6b", prompt=query)["embedding"]
    
    results = collection.query(query_embeddings=[query_emb], n_results=5)
    
    context = "\n\n".join(results["documents"][0])
    
    # Change the prompt if model is misbehaving
    prompt = f"""
    You are a precise teaching assistant.

    Answer the question using ONLY the context below.

    Give a clear and structured explanation.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    #print(context) #uncomment this if you think something might be wrong with the retrieval

    output = ollama.generate(model="mistral", prompt=prompt)
    return output['response']

print(local_rag("What counts as a tree and what counts as a forest?"))

 A **tree** is an undirected or directed graph that has at least one vertex, is connected, and has no closed trails with at least one edge. In other words, it's either a single path connecting all vertices or a collection of such paths that don't share any internal vertices (i.e., vertices other than the start and endpoints).

A **forest** is a collection of trees. If we have an undirected graph where every connected component is a tree, then that graph counts as a forest. In other words, it's a disjoint union of paths.

Note that by convention, a null graph (with no vertices or edges) is considered to be a forest with 0 trees. A tree can't have zero vertices because it must have at least one vertex.

In the given context, since we are dealing with an undirected graph with 6 trees, it counts as a forest. Each of these trees could represent individual paths that don't share any internal vertices.


In [ ]:
###IMPORTANT: This will clear your vectorstore, so DONT RUN ALL if you're planning to test multiple times or you will have to wait. Only clear the vectorstore if you are:
### 1. Adding new data
### 2. Trying a different chunking method/embedding model
### 3. Changing data/metadata structure

client = chromadb.Client()

#run this to clear vectorstore
try:
    client.delete_collection(name="local_pdf_rag")
except:
    pass

collection = client.get_or_create_collection(name="local_pdf_rag")